# **Build A Simple RAG From Scratch**

This notebook demonstrates how to build a simple Retrieval-Augmented Generation (RAG) system from scratch using Ollama. It covers the following steps:

**1. Loading the dataset:** Cat facts are loaded from a text file.

**2. Implementing the vector database:** Each fact is converted into an embedding vector using Ollama's hf.co/CompendiumLabs/bge-base-en-v1.5-gguf model and stored in a VECTOR_DB.

**3. Implementing the retrieval function:** A retrieve function is created to find the most similar cat facts to a given query using cosine similarity.

**4. Generation phase:** The retrieved facts are used as context in a prompt for a language model (hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF) to generate a response to the user's query.

In [ ]:
pip install ollama

### 1. Load dataset

In [ ]:
# Load dataset
dataset = []
with open('cat-facts.txt', 'r') as file:
  dataset = file.readlines()
  print(f'Loaded {len(dataset)} entries')

# Print five sentences of cat-facts.txt
dataset[:5]

Loaded 150 entries


['On average, cats spend 2/3 of every day sleeping. That means a nine-year-old cat has been awake for only three years of its life.\n',
 'Unlike dogs, cats do not have a sweet tooth. Scientists believe this is due to a mutation in a key taste receptor.\n',
 'When a cat chases its prey, it keeps its head level. Dogs and humans bob their heads up and down.\n',
 'The technical term for a cat’s hairball is a “bezoar.”\n',
 'A group of cats is called a “clowder.”\n']

### 2. Implement the vector database

#### An example function to calculate the embedding vector for a given text:

In [ ]:
import ollama

EMBEDDING_MODEL = 'hf.co/CompendiumLabs/bge-base-en-v1.5-gguf'
LANGUAGE_MODEL = 'hf.co/bartowski/Llama-3.2-1B-Instruct-GGUF'

# Each element in the VECTOR_DB will be a tuple (chunk, embedding)
# The embedding is a list of floats, for example: [0.1, 0.04, -0.34, 0.21, ...]
VECTOR_DB = []

def add_chunk_to_database(chunk):
  embedding = ollama.embed(model=EMBEDDING_MODEL, input=chunk)['embeddings'][0]
  VECTOR_DB.append((chunk, embedding))


#### We will consider each line in the dataset as a chunk for simplicity.

In [ ]:
for i, chunk in enumerate(dataset):
  add_chunk_to_database(chunk)
  print(f'Added chunk {i+1}/{len(dataset)} to the database')


Added chunk 1/150 to the database
Added chunk 2/150 to the database
Added chunk 3/150 to the database
Added chunk 4/150 to the database
Added chunk 5/150 to the database
Added chunk 6/150 to the database
Added chunk 7/150 to the database
Added chunk 8/150 to the database
Added chunk 9/150 to the database
Added chunk 10/150 to the database
Added chunk 11/150 to the database
Added chunk 12/150 to the database
Added chunk 13/150 to the database
Added chunk 14/150 to the database
Added chunk 15/150 to the database
Added chunk 16/150 to the database
Added chunk 17/150 to the database
Added chunk 18/150 to the database
Added chunk 19/150 to the database
Added chunk 20/150 to the database
Added chunk 21/150 to the database
Added chunk 22/150 to the database
Added chunk 23/150 to the database
Added chunk 24/150 to the database
Added chunk 25/150 to the database
Added chunk 26/150 to the database
Added chunk 27/150 to the database
Added chunk 28/150 to the database
Added chunk 29/150 to the dat


### 3. Implement the retrieval function

#### An example function to calculate the cosine similarity between two vectors:

In [ ]:
def cosine_similarity(a, b):
  dot_product = sum([x * y for x, y in zip(a, b)])
  norm_a = sum([x ** 2 for x in a]) ** 0.5
  norm_b = sum([x ** 2 for x in b]) ** 0.5
  return dot_product / (norm_a * norm_b)


#### Now, we implement the retrieval function.

In [ ]:
def retrieve(query, top_n=3):
  query_embedding = ollama.embed(model=EMBEDDING_MODEL, input=query)['embeddings'][0]
  # temporary list to store (chunk, similarity) pairs
  similarities = []
  for chunk, embedding in VECTOR_DB:
    similarity = cosine_similarity(query_embedding, embedding)
    similarities.append((chunk, similarity))
  # sort by similarity in descending order, because higher similarity means more relevant chunks
  similarities.sort(key=lambda x: x[1], reverse=True)
  # finally, return the top N most relevant chunks
  return similarities[:top_n]


### 4. Generation phrase

#### Construct a prompt

In [ ]:
input_query = input('Ask me a question: ')
retrieved_knowledge = retrieve(input_query)

print('Retrieved knowledge:')
for chunk, similarity in retrieved_knowledge:
  print(f' - (similarity: {similarity:.2f}) {chunk}')

# FIX: build this BEFORE the f-string
context_block = "\n".join([f" - {chunk}" for chunk, similarity in retrieved_knowledge])

instruction_prompt = f'''You are a helpful chatbot.
Use only the following pieces of context to answer the question. Don't make up any new information:
{context_block}
'''


Ask me a question:  where is cat live ?


Retrieved knowledge:
 - (similarity: 0.66) The oldest cat on record was Crème Puff from Austin, Texas, who lived from 1967 to August 6, 2005, three days after her 38th birthday. A cat typically can live up to 20 years, which is equivalent to about 96 human years.

 - (similarity: 0.65) Cats are North America’s most popular pets: there are 73 million cats compared to 63 million dogs. Over 30% of households in North America own a cat.

 - (similarity: 0.64) When well treated, a cat can live twenty or more years but the average life span of a domestic cat is 14 years.



#### We then use the ollama to generate the response. In this example, we will use instruction_prompt as system message:

In [ ]:
stream = ollama.chat(
  model=LANGUAGE_MODEL,
  messages=[
    {'role': 'system', 'content': instruction_prompt},
    {'role': 'user', 'content': input_query},
  ],
  stream=True,
)

# print the response from the chatbot in real-time
print('Chatbot response:')
for chunk in stream:
  print(chunk['message']['content'], end='', flush=True)

Chatbot response:
Unfortunately, I don't have any information about where cats typically live. However, based on my previous text, it seems that Crème Puff from Austin, Texas lived in North America. If you'd like to know more about her life or environment, feel free to ask!

### Reference
[1] RAG tutorials: https://huggingface.co/blog/ngxson/make-your-own-rag

